# The Urdu Tokenizer Tax
**How much more do LLMs "charge" for Urdu and Roman Urdu than for English, and does it hurt model quality?**

**How to run:** `Runtime → Run all`. Takes about 10 minutes. Part A (tokenizers) needs no GPU. Part B (small model quality test) uses a GPU if available (T4 is fine) and falls back to CPU.

**Outputs** (saved to `/content/results/`): `tokenizer_stats.csv`, `tokenizer_cost_chart.png`, `model_quality.csv`, `model_quality_chart.png`.

**Part A:** compare tokens needed for the *same sentences* in English, Urdu, and Roman Urdu across popular tokenizers.
**Part B:** check whether a small model is also worse at Urdu, measured in bits per byte (fair across languages), in full precision vs. 8-bit vs. 4-bit.

## 0. Install

In [ ]:
!pip install -q transformers datasets tiktoken sentencepiece protobuf accelerate bitsandbytes matplotlib pandas

## 1. Settings

In [ ]:
import os, gc, math, warnings
warnings.filterwarnings("ignore")
import pandas as pd
import matplotlib.pyplot as plt

OUT = "/content/results"
os.makedirs(OUT, exist_ok=True)

# Tokenizers to compare (any that fail to load are skipped automatically)
HF_TOKENIZERS = {
    "GPT-2": "gpt2",
    "Qwen2.5": "Qwen/Qwen2.5-0.5B",
    "BLOOM": "bigscience/bloom-560m",
    "XLM-R": "xlm-roberta-base",
    "mBERT": "bert-base-multilingual-cased",
    "mT5": "google/mt5-small",
    "Llama-2 (TinyLlama)": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "Phi-3": "microsoft/Phi-3-mini-4k-instruct",
    "XGLM": "facebook/xglm-564M",
}
TIKTOKEN_ENCODINGS = {"GPT-4 (cl100k)": "cl100k_base", "GPT-4o (o200k)": "o200k_base"}

RUN_PART_B = True                 # set False to run only the tokenizer study
MODEL_FOR_PART_B = "Qwen/Qwen2.5-0.5B"
N_EVAL = 100                      # sentences per language for Part B

## 2. Data: 20 hand-written parallel sentences (English / Urdu / Roman Urdu) + optional OPUS-100

In [ ]:
curated = [
 ("The weather is very pleasant today.", "آج موسم بہت خوشگوار ہے۔", "Aaj mausam bohat khushgawar hai."),
 ("I am going to the market to buy fruits.", "میں پھل خریدنے بازار جا رہا ہوں۔", "Main phal kharidne bazaar ja raha hoon."),
 ("Education is the key to a better future.", "تعلیم بہتر مستقبل کی کنجی ہے۔", "Taleem behtar mustaqbil ki kunji hai."),
 ("Islamabad is the capital of Pakistan.", "اسلام آباد پاکستان کا دارالحکومت ہے۔", "Islamabad Pakistan ka daar-ul-hukumat hai."),
 ("Please close the door when you leave.", "جاتے وقت براہ مہربانی دروازہ بند کر دیں۔", "Jaate waqt barah-e-meharbani darwaza band kar dein."),
 ("She reads a book every evening.", "وہ ہر شام ایک کتاب پڑھتی ہے۔", "Woh har shaam ek kitaab parhti hai."),
 ("The students are preparing for their exams.", "طلبہ اپنے امتحانات کی تیاری کر رہے ہیں۔", "Talba apne imtihanaat ki tayyari kar rahe hain."),
 ("Technology is changing the way we live.", "ٹیکنالوجی ہمارے جینے کا انداز بدل رہی ہے۔", "Technology hamare jeene ka andaaz badal rahi hai."),
 ("We should respect our elders.", "ہمیں اپنے بزرگوں کا احترام کرنا چاہیے۔", "Hamein apne buzurgon ka ehtiram karna chahiye."),
 ("The train will arrive at the station in ten minutes.", "ٹرین دس منٹ میں اسٹیشن پر پہنچ جائے گی۔", "Train das minute mein station par pahunch jayegi."),
 ("Honesty is the best policy.", "ایمانداری بہترین حکمت عملی ہے۔", "Imaandaari behtareen hikmat-e-amali hai."),
 ("He works as a software engineer in Lahore.", "وہ لاہور میں سافٹ ویئر انجینئر کے طور پر کام کرتا ہے۔", "Woh Lahore mein software engineer ke taur par kaam karta hai."),
 ("Rain is expected in the northern areas tomorrow.", "کل شمالی علاقوں میں بارش کی توقع ہے۔", "Kal shimali ilaqon mein barish ki tawaqqo hai."),
 ("Debate teaches us to listen as well as to speak.", "مباحثہ ہمیں بولنے کے ساتھ ساتھ سننا بھی سکھاتا ہے۔", "Mubahisa hamein bolne ke saath saath sunna bhi sikhata hai."),
 ("The library is open from nine to five.", "لائبریری صبح نو بجے سے شام پانچ بجے تک کھلی رہتی ہے۔", "Library subah nau bajay se shaam paanch bajay tak khuli rehti hai."),
 ("Clean water is essential for good health.", "اچھی صحت کے لیے صاف پانی ضروری ہے۔", "Achi sehat ke liye saaf pani zaroori hai."),
 ("My brother is learning to play the guitar.", "میرا بھائی گٹار بجانا سیکھ رہا ہے۔", "Mera bhai guitar bajana seekh raha hai."),
 ("The government announced a new education policy.", "حکومت نے نئی تعلیمی پالیسی کا اعلان کیا۔", "Hukumat ne nayi taleemi policy ka elaan kiya."),
 ("Time is more valuable than money.", "وقت پیسے سے زیادہ قیمتی ہے۔", "Waqt paise se zyada qeemti hai."),
 ("Let us work together to solve this problem.", "آئیں مل کر اس مسئلے کو حل کریں۔", "Aaiye mil kar is masle ko hal karein."),
]
cur_en  = [x[0] for x in curated]
cur_ur  = [x[1] for x in curated]
cur_rom = [x[2] for x in curated]

# Optional larger real-world parallel data (noisy: subtitles/religious/web text). Skipped if unavailable.
opus_en, opus_ur = [], []
try:
    from datasets import load_dataset
    ds = load_dataset("Helsinki-NLP/opus-100", "en-ur", split="test")
    for ex in ds:
        e, u = ex["translation"]["en"].strip(), ex["translation"]["ur"].strip()
        if 30 <= len(e) <= 250 and 30 <= len(u) <= 250 and len(e.split()) >= 5:
            opus_en.append(e); opus_ur.append(u)
        if len(opus_en) >= 500:
            break
    print(f"Loaded {len(opus_en)} OPUS-100 en-ur pairs")
except Exception as ex:
    print("OPUS-100 not available, continuing with the curated set only ->", str(ex)[:100])

DATASETS = {"Curated (20 sentences)": {"English": cur_en, "Urdu": cur_ur, "Roman Urdu": cur_rom}}
if opus_en:
    DATASETS[f"OPUS-100 ({len(opus_en)} pairs)"] = {"English": opus_en, "Urdu": opus_ur}
print(list(DATASETS))

## 3. Part A: Load tokenizers

In [ ]:
from transformers import AutoTokenizer
import tiktoken

counters = {}
for label, repo in HF_TOKENIZERS.items():
    try:
        tk = AutoTokenizer.from_pretrained(repo)
        counters[label] = (lambda t, tk=tk: len(tk.encode(t, add_special_tokens=False)))
        print("OK   ", label)
    except Exception as e:
        print("SKIP ", label, "->", str(e)[:90])

for label, enc in TIKTOKEN_ENCODINGS.items():
    try:
        e = tiktoken.get_encoding(enc)
        counters[label] = (lambda t, e=e: len(e.encode(t, disallowed_special=())))
        print("OK   ", label)
    except Exception as ex:
        print("SKIP ", label, "->", str(ex)[:90])

print(f"\n{len(counters)} tokenizers ready")

## 4. Part A: Measure the tokenizer tax

In [ ]:
rows = []
for ds_name, langs in DATASETS.items():
    for tname, count in counters.items():
        for lang, texts in langs.items():
            tokens = sum(count(t) for t in texts)
            words  = sum(len(t.split()) for t in texts)
            nbytes = sum(len(t.encode("utf-8")) for t in texts)
            rows.append(dict(dataset=ds_name, tokenizer=tname, language=lang, n_sentences=len(texts),
                             tokens=tokens, tokens_per_word=tokens / words, bytes_per_token=nbytes / tokens))
df = pd.DataFrame(rows)

# "Tax" = how many times more tokens the same content needs vs English
en_tokens = df[df.language == "English"].set_index(["dataset", "tokenizer"])["tokens"]
df["cost_vs_english"] = df.apply(lambda r: r.tokens / en_tokens[(r.dataset, r.tokenizer)], axis=1)

df.round(3).to_csv(f"{OUT}/tokenizer_stats.csv", index=False)
pd.set_option("display.width", 200)
for ds_name in DATASETS:
    print(f"\n=== {ds_name} ===")
    view = df[df.dataset == ds_name].pivot(index="tokenizer", columns="language", values="cost_vs_english").round(2)
    print(view.sort_values("Urdu", ascending=False).to_string())

In [ ]:
n = len(DATASETS)
fig, axes = plt.subplots(1, n, figsize=(7 * n, 5.5), squeeze=False)
for ax, ds_name in zip(axes[0], DATASETS):
    sub = df[(df.dataset == ds_name) & (df.language != "English")]
    piv = sub.pivot(index="tokenizer", columns="language", values="cost_vs_english")
    piv = piv.sort_values("Urdu")
    piv.plot.barh(ax=ax, width=0.75)
    ax.axvline(1, color="k", ls="--", lw=1)
    ax.set_title(ds_name)
    ax.set_xlabel("Tokens needed vs. English (1.0 = same as English)")
    ax.set_ylabel("")
fig.suptitle("The tokenizer tax: same meaning, how many more tokens?", fontsize=13)
plt.tight_layout()
plt.savefig(f"{OUT}/tokenizer_cost_chart.png", dpi=160)
plt.show()

## 5. Part B: Does the model get worse at Urdu too? (bits per byte, full precision vs 8-bit vs 4-bit)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

has_gpu = torch.cuda.is_available()
print("GPU available:", has_gpu)

eval_en = (cur_en + opus_en)[:N_EVAL]
eval_ur = (cur_ur + opus_ur)[:N_EVAL]
print(len(eval_en), "English and", len(eval_ur), "Urdu sentences for evaluation")

@torch.no_grad()
def bits_per_byte(model, tok, texts, device):
    # Total negative log-likelihood (in bits) divided by total UTF-8 bytes.
    # Bytes, not tokens, so English and Urdu are comparable despite different tokenization.
    start = tok.bos_token_id if tok.bos_token_id is not None else tok.eos_token_id
    total_nll, total_bytes = 0.0, 0
    for t in texts:
        ids = tok(t, add_special_tokens=False, return_tensors="pt").input_ids
        if ids.shape[1] > 500:
            continue
        ids = torch.cat([torch.tensor([[start]]), ids], dim=1).to(device)
        out = model(ids, labels=ids)
        n_pred = ids.shape[1] - 1
        total_nll += out.loss.item() * n_pred
        total_bytes += len(t.encode("utf-8"))
    return total_nll / (math.log(2) * total_bytes)

In [ ]:
if not RUN_PART_B:
    print("Part B skipped (RUN_PART_B = False)")
else:
    tokB = AutoTokenizer.from_pretrained(MODEL_FOR_PART_B)
    configs = {}
    if has_gpu:
        configs["fp16"] = dict(torch_dtype=torch.float16, device_map="cuda")
        configs["8-bit"] = dict(quantization_config=BitsAndBytesConfig(load_in_8bit=True), device_map="cuda")
        configs["4-bit (NF4)"] = dict(quantization_config=BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16), device_map="cuda")
    else:
        print("No GPU: only the full-precision baseline can run. Use Runtime -> Change runtime type -> T4 GPU for 8-bit/4-bit.")
        configs["fp32 (cpu)"] = dict(torch_dtype=torch.float32)

    results = []
    for cname, kw in configs.items():
        try:
            model = AutoModelForCausalLM.from_pretrained(MODEL_FOR_PART_B, **kw).eval()
            dev = next(model.parameters()).device
            for lang, texts in {"English": eval_en, "Urdu": eval_ur}.items():
                results.append(dict(model=MODEL_FOR_PART_B, precision=cname, language=lang,
                                    bits_per_byte=bits_per_byte(model, tokB, texts, dev)))
            print("done:", cname)
        except Exception as e:
            print("FAILED:", cname, "->", str(e)[:150])
        finally:
            try: del model
            except NameError: pass
            gc.collect()
            if has_gpu: torch.cuda.empty_cache()

    qdf = pd.DataFrame(results)
    if not qdf.empty:
        base_name = qdf.precision.iloc[0]
        base = qdf[qdf.precision == base_name].set_index("language")["bits_per_byte"]
        qdf["pct_worse_than_baseline"] = qdf.apply(
            lambda r: 100 * (r.bits_per_byte - base[r.language]) / base[r.language], axis=1)
        qdf.round(4).to_csv(f"{OUT}/model_quality.csv", index=False)
        print(qdf.round(4).to_string(index=False))

        piv = qdf.pivot(index="precision", columns="language", values="bits_per_byte")
        piv = piv.reindex([c for c in configs if c in piv.index])
        piv.plot.bar(figsize=(7, 4.5), rot=0)
        plt.ylabel("Bits per byte (lower is better)")
        plt.title(f"{MODEL_FOR_PART_B}: quality by precision and language")
        plt.tight_layout()
        plt.savefig(f"{OUT}/model_quality_chart.png", dpi=160)
        plt.show()

## 6. Summary: your findings, auto-filled

In [ ]:
primary = list(DATASETS)[-1]
sub = df[(df.dataset == primary) & (df.language == "Urdu")].sort_values("cost_vs_english")
best, worst = sub.iloc[0], sub.iloc[-1]
print(f"Dataset used for headline numbers: {primary}\n")
print(f"- Urdu needs between {best.cost_vs_english:.2f}x ({best.tokenizer}) and {worst.cost_vs_english:.2f}x ({worst.tokenizer}) as many tokens as English.")
if "Roman Urdu" in df.language.values:
    r = df[(df.dataset == list(DATASETS)[0]) & (df.language == "Roman Urdu")]
    print(f"- Roman Urdu (curated set) ranges from {r.cost_vs_english.min():.2f}x to {r.cost_vs_english.max():.2f}x vs English.")
print(f"- Fewest tokens per Urdu word: {sub.sort_values('tokens_per_word').iloc[0].tokenizer}; most: {sub.sort_values('tokens_per_word').iloc[-1].tokenizer}.")
try:
    q = qdf.set_index(["precision", "language"])["bits_per_byte"]
    for p in qdf.precision.unique():
        print(f"- {p}: English {q[(p,'English')]:.3f} bpb, Urdu {q[(p,'Urdu')]:.3f} bpb")
except Exception:
    pass

# Download everything (works in Colab; ignored elsewhere)
try:
    import shutil
    shutil.make_archive("/content/results_bundle", "zip", OUT)
    from google.colab import files
    files.download("/content/results_bundle.zip")
except Exception:
    print("\nFiles are in", OUT)

## 7. Put it on GitHub
1. Create a repo `urdu-tokenizer-tax`. In Colab: **File → Save a copy in GitHub**.
2. Add `results/` (the CSVs and PNGs from the zip) to the repo.
3. In your README, include: the question, the headline chart, the table from Part A, the Part B table, and **limitations** (small curated set, OPUS-100 is noisy, a 0.5B model only, bits per byte computed on short sentences).
4. Optional upgrades: add more models for Part B, add Punjabi/Pashto/Sindhi sentences, or write a blog post on your best chart.